<a href="https://colab.research.google.com/github/CarlosJB95/PathIA-MSI-colon/blob/main/Notebook_Semana_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Semana 2 · Día 6 — NumPy sobre parches H&E**
Objetivo: tratar un parche como ndarray (alto, ancho, 3): crear, indexar, cortar, operar.
Cierra el Notebook 1 (Python intermedio). Arrastre a reabsorber hoy en la tarde: try/except + escribir manifest a CSV.

In [1]:
import numpy as np
np.random.seed(42)   # misma "aleatoriedad" siempre → reproducible

In [2]:
a = np.array([200], dtype=np.uint8)
print(a + 100)     # ¿300? No... → 44   (200+100 = 300; 300 - 256 = 44)

b = np.array([10], dtype=np.uint8)
print(b - 50)      # ¿-40? No... → 216  (da la vuelta por abajo)

[44]
[216]


In [3]:
np.array([1, 2, 3])        # a partir de datos que YA tienes (una lista)
np.zeros((8, 8, 3))        # lienzo en blanco: todo 0.0  → reservar espacio
np.ones((8, 8, 3))         # todo 1.0                     → máscaras, pesos
np.arange(0, 10, 2)        # rango con paso → [0 2 4 6 8]  (como range de Python)
np.linspace(0, 1, 5)       # N puntos equiespaciados → [0. 0.25 0.5 0.75 1.]

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

In [4]:
x = np.zeros((8, 8, 3))
print("shape:", x.shape)   # (8, 8, 3)  → tamaño por eje. TU dato más importante.
print("ndim :", x.ndim)    # 3          → número de ejes (dimensiones)
print("dtype:", x.dtype)   # float64    → tipo de cada elemento (default al no especificar)
print("size :", x.size)    # 192        → total de elementos = 8×8×3

shape: (8, 8, 3)
ndim : 3
dtype: float64
size : 192


In [5]:
parche = np.random.randint(0, 256, size=(8, 8, 3), dtype=np.uint8)
print(parche.shape, parche.ndim, parche.dtype)   # (8, 8, 3) 3 uint8  ← tupla; uint8 porque lo fijé en el constructor

(8, 8, 3) 3 uint8


In [6]:
print("R medio:", parche[:, :, 0].mean())
print("G medio:", parche[:, :, 1].mean())
print("B medio:", parche[:, :, 2].mean())

R medio: 140.390625
G medio: 139.734375
B medio: 118.65625


In [7]:
# Parche 8x8: mitad izquierda "citoplasma" (eosina), mitad derecha "núcleo" (hematoxilina)
parche_he = np.zeros((8, 8, 3), dtype=np.uint8)

# Eosina (rosa): R alto, G/B medios-bajos  -> columnas 0-3
parche_he[:, 0:4] = [230, 130, 180]

# Hematoxilina (azul-morado): B alto, R medio, G bajo -> columnas 4-7
parche_he[:, 4:8] = [110, 90, 200]

# Media por canal de CADA región
print("Eosina  (izq):", parche_he[:, 0:4].mean(axis=(0,1)))
print("Hematox (der):", parche_he[:, 4:8].mean(axis=(0,1)))

Eosina  (izq): [230. 130. 180.]
Hematox (der): [110.  90. 200.]


In [8]:
print(parche_he.mean().shape or "escalar")   # ()  → escalar, sin ejes
print(parche_he.mean(axis=(0,1)).shape)       # (3,)
print(parche_he.mean(axis=2).shape)           # (8, 8)

escalar
(3,)
(8, 8)


In [9]:
p = np.zeros((256, 256, 3), dtype=np.uint8)
print("entrada:   ", p.shape)              # (256, 256, 3)
print("axis=(0,1):", p.mean(axis=(0,1)).shape)   # tacho 0 y 1 → (3, )
print("axis=2:    ", p.mean(axis=2).shape)       # tacho 2     → (256, 256)
print("axis=0:    ", p.mean(axis=0).shape)       # tacho 0     → (256, 3)

entrada:    (256, 256, 3)
axis=(0,1): (3,)
axis=2:     (256, 256)
axis=0:     (256, 3)


In [10]:
parche_f = parche_he.astype(np.float32)      # a float primero (¿por qué? → q3 😉)
firma    = parche_f.mean(axis=(0,1))         # (3,)  → [media_R, media_G, media_B]

centrado = parche_f - firma                  # (8,8,3) - (3,)  ... ¿y funciona?
print(centrado.shape)                        # (8, 8, 3)

(8, 8, 3)


In [11]:
a = np.ones((8, 8, 3))
print((a * 2).shape)               # (i)   (8, 8, 3) — un escalar cambia los VALORES, no la forma
print((a - np.ones((3,))).shape)   # (ii)  (8, 8, 3) — resta por canal (último eje 3 == 3)
# a - np.ones((8,))                # (iii) truena: último eje 3 vs 8, y ninguno es 1

(8, 8, 3)
(8, 8, 3)


In [12]:
# 2 de las 8 filas = fondo (vidrio, casi blanco)
parche_he[0:2, :] = [245, 245, 245]

gris = parche_he.mean(axis=2)            # (a) (8, 8) → colapsa canales, conserva la rejilla espacial
mask_tejido = gris < 220                 # (b) bool  → máscara: True = tejido
pct_tejido = mask_tejido.mean() * 100    # (c) media de un booleano (True=1) = fracción de píxeles de tejido

print("shape máscara:", mask_tejido.shape)
print("% tejido:", pct_tejido)

shape máscara: (8, 8)
% tejido: 75.0


In [13]:
parche_f = parche_he.astype(np.float32)          # calcular en float (¿por qué? 😉)

print("media  RGB:", parche_f.mean(axis=(0,1)))  # firma de color
print("std    RGB:", parche_f.std(axis=(0,1)))   # dispersión por canal
print("min    RGB:", parche_f.min(axis=(0,1)))
print("max    RGB:", parche_f.max(axis=(0,1)))

# percentiles sobre UN canal (el rojo), para mediana e IQR
R = parche_f[:, :, 0]
p25, p50, p75 = np.percentile(R, [25, 50, 75])
print(f"\nCanal R → mediana={p50:.0f}, IQR={p75-p25:.0f}")

media  RGB: [188.75 143.75 203.75]
std    RGB: [61.275505 60.968742 25.34142 ]
min    RGB: [110.  90. 180.]
max    RGB: [245. 245. 245.]

Canal R → mediana=230, IQR=124


In [14]:
def describir_parche(patch, slide_id, patch_id):
    """Extrae las métricas de un parche H&E y las devuelve como dict (una fila del manifest)."""
    gris = patch.mean(axis=2)
    pct_tejido = float((gris < 220).mean() * 100)      # ← float() añadido: cruza la aduana
    media_rgb = patch.astype(np.float32).mean(axis=(0, 1))
    return {
        "slide_id": slide_id,
        "patch_id": patch_id,
        "pct_tejido": round(pct_tejido, 1),
        "media_R": round(float(media_rgb[0]), 1),
        "media_G": round(float(media_rgb[1]), 1),
        "media_B": round(float(media_rgb[2]), 1),
    }

fila = describir_parche(parche_he, slide_id="TCGA-XX-0001", patch_id="p000")
print(fila)

{'slide_id': 'TCGA-XX-0001', 'patch_id': 'p000', 'pct_tejido': 75.0, 'media_R': 188.8, 'media_G': 143.8, 'media_B': 203.8}


In [15]:
import csv

def escribir_manifest(filas, ruta):
    """Escribe una lista de dicts (filas del manifest) a un CSV. Maneja errores de E/S sin abortar."""
    if not filas:
        print("⚠️  No hay filas que escribir.")
        return False
    columnas = filas[0].keys()
    try:
        with open(ruta, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=columnas)
            writer.writeheader()
            writer.writerows(filas)
    except (OSError, PermissionError) as e:
        print(f"❌ No se pudo escribir '{ruta}': {e}")
        return False
    else:
        print(f"✅ Manifest escrito: {ruta}  ({len(filas)} filas)")
        return True

In [16]:
# 3 filas: el parche real + dos variantes con distinto % de fondo
p_mixto = parche_he.copy()
p_fondo = parche_he.copy();  p_fondo[:6, :] = [245, 245, 245]   # casi todo fondo

filas = [
    describir_parche(p_mixto, "TCGA-XX-0001", "p000"),
    describir_parche(p_fondo, "TCGA-XX-0001", "p001"),
]

escribir_manifest(filas, "manifest_parches.csv")

✅ Manifest escrito: manifest_parches.csv  (2 filas)


True

In [17]:
# 1) Caso bueno
escribir_manifest(filas, "manifest_parches.csv")        # ✅ ... (2 filas)

# 2) Provoca el fallo a propósito
escribir_manifest(filas, "/carpeta_inexistente/manifest.csv")   # ❌ ... y NO muere

print("El programa sigue vivo después del error 👇")     # ← esta línea DEBE ejecutarse

✅ Manifest escrito: manifest_parches.csv  (2 filas)
❌ No se pudo escribir '/carpeta_inexistente/manifest.csv': [Errno 2] No such file or directory: '/carpeta_inexistente/manifest.csv'
El programa sigue vivo después del error 👇


## **Semana 2 · Día 7 — Operaciones vectorizadas y estadística**

Temas: Reducciones por eje, lógica booleana, np.where y estadística descriptiva (media/mediana/SD/percentiles/IQR/histograma) sobre el parche H&E

In [18]:
R = parche_he[:, :, 0].astype(np.int16)   # int16 para restar sin desborde de uint8
B = parche_he[:, :, 2].astype(np.int16)

es_tejido  = gris < 220        # ya la conoces: oscuro
es_azulado = B > R             # el canal azul supera al rojo → hematoxilina

nucleos = es_tejido & es_azulado    # tejido Y azulado

In [19]:
nucleos = (gris < 220) & (B > R)
print(nucleos.shape, nucleos.dtype)
print("% tejido :", (gris < 220).mean() * 100)
print("% núcleos:", nucleos.mean() * 100)

(8, 8) bool
% tejido : 75.0
% núcleos: 37.5


In [20]:
# Resaltar núcleos: 255 donde hay núcleo, 0 donde no → imagen binaria
resaltado = np.where(nucleos, 255, 0)
print(resaltado.shape, resaltado.dtype)
print(resaltado)

(8, 8) int64
[[  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0 255 255 255 255]
 [  0   0   0   0 255 255 255 255]
 [  0   0   0   0 255 255 255 255]
 [  0   0   0   0 255 255 255 255]
 [  0   0   0   0 255 255 255 255]
 [  0   0   0   0 255 255 255 255]]


In [21]:
# Versión 2D (binaria) — pregunta 1
resaltado = np.where(nucleos, 255, 0)
print("binaria:", resaltado.shape)          # (8, 8)

# Versión a color — pregunta 2, ya arreglada
verde = np.array([0, 255, 0], dtype=np.uint8)
coloreado = np.where(nucleos[:, :, None], verde, parche_he)
print("color:  ", coloreado.shape)          # (8, 8, 3)

binaria: (8, 8)
color:   (8, 8, 3)


In [22]:
def estadisticas_parche(patch, umbral_tejido=220):
    """Describe un parche H&E: % tejido, % núcleos y firma de color por canal.
    Devuelve un dict (una fila de manifest). Solo calcula; no toca disco."""
    gris = patch.mean(axis=2)
    R = patch[:, :, 0].astype(np.int16)
    B = patch[:, :, 2].astype(np.int16)

    es_tejido = gris < umbral_tejido
    nucleos   = es_tejido & (B > R)
    media_rgb = patch.astype(np.float32).mean(axis=(0, 1))

    return {
        "pct_tejido":  round(float(es_tejido.mean() * 100), 1),
        "pct_nucleos": round(float(nucleos.mean() * 100), 1),
        "media_R":     round(float(media_rgb[0]), 1),
        "media_G":     round(float(media_rgb[1]), 1),
        "media_B":     round(float(media_rgb[2]), 1),
    }

print(estadisticas_parche(parche_he))

{'pct_tejido': 75.0, 'pct_nucleos': 37.5, 'media_R': 188.8, 'media_G': 143.8, 'media_B': 203.8}


In [23]:
def pct_tejido_lento(patch, umbral=220):
    """Versión con bucles: recorre cada píxel a mano. LENTA a propósito."""
    H, W, _ = patch.shape
    cuenta = 0
    for i in range(H):
        for j in range(W):
            gris_px = (int(patch[i,j,0]) + int(patch[i,j,1]) + int(patch[i,j,2])) / 3
            if gris_px < umbral:
                cuenta += 1
    return cuenta / (H * W) * 100

# La versión vectorizada (una línea, todo NumPy):
def pct_tejido_rapido(patch, umbral=220):
    return (patch.mean(axis=2) < umbral).mean() * 100

In [24]:
grande = np.random.randint(0, 256, size=(256, 256, 3), dtype=np.uint8)

# PASO 1 — ¿dan lo mismo? (corrección antes que velocidad)
print(pct_tejido_lento(grande), pct_tejido_rapido(grande))   # deben coincidir

98.81744384765625 98.81744384765625


In [25]:
%%timeit
pct_tejido_lento(grande)

139 ms ± 52.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [26]:
%%timeit
pct_tejido_rapido(grande)

2.03 ms ± 59.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [27]:
gris = parche.mean(axis=2)
pct_fondo = (gris > 230).mean() * 100
print(round(float(pct_fondo), 1))

0.0


In [28]:
def filtrar_parche(patch, umbral=220):
    """Describe un parche H&E y decide si se conserva. Solo calcula; no toca disco."""
    gris = patch.mean(axis=2)
    es_tejido = gris < umbral
    pct = float(es_tejido.mean() * 100)
    m = patch.astype(np.float32).mean(axis=(0, 1))
    return {
        "pct_tejido": round(pct, 1),
        "media_R": round(float(m[0]), 1),
        "media_G": round(float(m[1]), 1),
        "media_B": round(float(m[2]), 1),
        "conservar": pct >= 50,
    }

print(filtrar_parche(parche))

{'pct_tejido': 98.4, 'media_R': 140.4, 'media_G': 139.7, 'media_B': 118.7, 'conservar': True}


In [29]:
# 3 parches con distinto % de fondo, para variar
p0 = parche.copy()
p1 = parche.copy(); p1[:5, :] = 245     # mayoría fondo
p2 = parche.copy(); p2[:2, :] = 245     # poco fondo

filas = [filtrar_parche(p, umbral=220) for p in [p0, p1, p2]]
for f in filas:
    print(f)

{'pct_tejido': 98.4, 'media_R': 140.4, 'media_G': 139.7, 'media_B': 118.7, 'conservar': True}
{'pct_tejido': 35.9, 'media_R': 209.4, 'media_G': 203.6, 'media_B': 199.1, 'conservar': False}
{'pct_tejido': 73.4, 'media_R': 166.0, 'media_G': 165.0, 'media_B': 149.2, 'conservar': True}


# **Semana 2 · Día 8 — pandas sobre el manifest**
Objetivo: cargar/crear el manifest como DataFrame, explorarlo (head, dtypes, describe) y filtrarlo.
Puente: una lista de dicts (Notebook 02) → un DataFrame (tabla de la cohorte).

---



In [30]:
import numpy as np
import pandas as pd
np.random.seed(42)

def filtrar_parche(patch, umbral=220):
    gris = patch.mean(axis=2)
    pct  = float((gris < umbral).mean() * 100)
    m    = patch.astype(np.float32).mean(axis=(0, 1))
    return {"pct_tejido": round(pct, 1),
            "media_R": round(float(m[0]), 1),
            "media_G": round(float(m[1]), 1),
            "media_B": round(float(m[2]), 1),
            "conservar": pct >= 50}

filas = []
for k in range(12):
    p = np.random.randint(0, 256, size=(8, 8, 3), dtype=np.uint8)
    n_fondo = k % 8                       # 0 a 7 filas de fondo → variedad
    if n_fondo:
        p[:n_fondo, :] = 245
    fila = {"slide_id": f"TCGA-{k // 4:02d}", "patch_id": f"p{k:03d}", **filtrar_parche(p)}
    filas.append(fila)

df = pd.DataFrame(filas)     # ← EL PUENTE: lista de dicts → DataFrame
df

,slide_id,patch_id,pct_tejido,media_R,media_G,media_B,conservar
0,TCGA-00,p000,98.4,140.4,139.7,118.7,True
1,TCGA-00,p001,85.9,132.0,146.3,125.3,True
2,TCGA-00,p002,73.4,149.7,159.0,143.5,True
3,TCGA-00,p003,62.5,163.7,176.0,172.9,True
4,TCGA-01,p004,50.0,180.9,184.1,186.2,True
5,TCGA-01,p005,37.5,199.5,203.7,202.2,False
6,TCGA-01,p006,25.0,221.7,206.9,213.9,False
7,TCGA-01,p007,12.5,229.6,236.1,230.6,False
8,TCGA-02,p008,98.4,130.5,135.2,134.0,True
9,TCGA-02,p009,87.5,147.4,140.5,142.3,True


In [31]:
print(df.shape)      # (12, 7)
df.head()            # primeras 5 filas — como asomarte a la laminilla antes de contarla toda

(12, 7)


,slide_id,patch_id,pct_tejido,media_R,media_G,media_B,conservar
0,TCGA-00,p000,98.4,140.4,139.7,118.7,True
1,TCGA-00,p001,85.9,132.0,146.3,125.3,True
2,TCGA-00,p002,73.4,149.7,159.0,143.5,True
3,TCGA-00,p003,62.5,163.7,176.0,172.9,True
4,TCGA-01,p004,50.0,180.9,184.1,186.2,True


In [32]:
df.describe()

,pct_tejido,media_R,media_G,media_B
count,12.000000,12.000000,12.000000,12.000000
mean,64.050000,168.275000,170.208333,167.208333
std,28.097088,33.335979,31.760280,36.068709
min,12.500000,130.500000,135.200000,118.700000
25%,46.875000,145.650000,144.850000,140.225000
50%,67.950000,159.800000,158.500000,167.200000
75%,86.300000,185.550000,189.000000,190.200000
max,98.400000,229.600000,236.100000,230.600000


In [33]:
df["conservar"]          # una Serie de True/False, una por fila → ¡es una máscara booleana!
df[df["conservar"]]      # quédate solo con las filas donde la máscara es True

,slide_id,patch_id,pct_tejido,media_R,media_G,media_B,conservar
0,TCGA-00,p000,98.4,140.4,139.7,118.7,True
1,TCGA-00,p001,85.9,132.0,146.3,125.3,True
2,TCGA-00,p002,73.4,149.7,159.0,143.5,True
3,TCGA-00,p003,62.5,163.7,176.0,172.9,True
4,TCGA-01,p004,50.0,180.9,184.1,186.2,True
8,TCGA-02,p008,98.4,130.5,135.2,134.0,True
9,TCGA-02,p009,87.5,147.4,140.5,142.3,True
10,TCGA-02,p010,75.0,155.9,157.0,161.5,True
11,TCGA-02,p011,62.5,168.0,158.0,175.4,True


In [34]:
conservables = df[df["pct_tejido"] >= 50]     # máscara al vuelo (mismo criterio)
print("Total parches:", len(df))
print("Conservables :", len(conservables))
descartados = len(df) - len(conservables)
print("Descartados  :", descartados)

Total parches: 12
Conservables : 9
Descartados  : 3


In [35]:
conservables.to_csv("manifest_conservables.csv", index=False)

# releer y confirmar que NO apareció ninguna columna 'Unnamed'
check = pd.read_csv("manifest_conservables.csv")
print(check.shape)        # (9, 7)  → 9 conservables, 7 columnas (sin basura)
print(check.columns.tolist())

(9, 7)
['slide_id', 'patch_id', 'pct_tejido', 'media_R', 'media_G', 'media_B', 'conservar']


In [36]:
df.iloc[:3]                    # (a) primeras 3 filas por POSICIÓN
df.loc[0, "pct_tejido"]        # (b) valor en etiqueta de fila 0, columna "pct_tejido"
df.loc[df.pct_tejido > 25]     # (c) filtrado booleano por etiqueta

,slide_id,patch_id,pct_tejido,media_R,media_G,media_B,conservar
0,TCGA-00,p000,98.4,140.4,139.7,118.7,True
1,TCGA-00,p001,85.9,132.0,146.3,125.3,True
2,TCGA-00,p002,73.4,149.7,159.0,143.5,True
3,TCGA-00,p003,62.5,163.7,176.0,172.9,True
4,TCGA-01,p004,50.0,180.9,184.1,186.2,True
5,TCGA-01,p005,37.5,199.5,203.7,202.2,False
8,TCGA-02,p008,98.4,130.5,135.2,134.0,True
9,TCGA-02,p009,87.5,147.4,140.5,142.3,True
10,TCGA-02,p010,75.0,155.9,157.0,161.5,True
11,TCGA-02,p011,62.5,168.0,158.0,175.4,True


In [37]:
print(df.iloc[:3].shape)         # (3, 7) → DataFrame
print(len(df.iloc[0:3]))         # 3  (exclusivo)  ✓ tu predicción
print(len(df.loc[0:3]))          # 4  (inclusivo)  ✓ tu predicción
print(df.loc[0, "pct_tejido"])   # un valor escalar

(3, 7)
3
4
98.4


In [38]:
solo_A     = df[df.pct_tejido > 25]
ambas      = df[(df.pct_tejido > 25) & (df.media_R < 150)]

print("solo pct_tejido>25:", len(solo_A))
print("ambas condiciones :", len(ambas))
print("cota A∩B ⊆ A:", len(ambas) <= len(solo_A))   # True, siempre

solo pct_tejido>25: 10
ambas condiciones : 5
cota A∩B ⊆ A: True


In [39]:
# Asignación: una comparación (máscara booleana) SE VUELVE una columna nueva.
# Usamos un nombre NUEVO para no pisar el criterio oficial `conservar` (pct >= 50).
df["conservar_25"] = df.pct_tejido > 25          # umbral alternativo (Kaggle L2)

print(df["conservar_25"].dtype)                  # bool
print((df.pct_tejido > 25).dtype)                # bool  ← una comparación SIEMPRE da bool
print(df.dtypes)                                 # cada columna en su casilla: object / float64 / bool

bool
bool
slide_id         object
patch_id         object
pct_tejido      float64
media_R         float64
media_G         float64
media_B         float64
conservar          bool
conservar_25       bool
dtype: object


In [40]:
# value_counts devuelve una Serie: el índice son las categorías, los valores son los conteos
print(df["conservar"].value_counts())   # True 9 / False 3  (criterio oficial pct>=50)
print()
print(df["slide_id"].value_counts())    # 4 parches por laminilla × 3 laminillas

conservar
True     9
False    3
Name: count, dtype: int64

slide_id
TCGA-00    4
TCGA-01    4
TCGA-02    4
Name: count, dtype: int64


In [41]:
df["slide_id"].unique()      # ¿QUÉ valores distintos hay? → array(['TCGA-00','TCGA-01','TCGA-02'])
df["slide_id"].nunique()     # ¿CUÁNTOS valores distintos hay? → 3

3

In [42]:
print("laminillas distintas :", df["slide_id"].nunique())      # 3
print("pct_tejido distintos :", df["pct_tejido"].nunique())    # ¿12? ¿o menos por empates?
print("total de filas       :", len(df))                       # 12

laminillas distintas : 3
pct_tejido distintos : 10
total de filas       : 12


In [43]:
df["nivel_tejido"] = df["pct_tejido"].map(
    lambda p: "alto" if p > 50 else "medio" if p > 25 else "bajo"
)
df[["patch_id", "pct_tejido", "nivel_tejido"]]

,patch_id,pct_tejido,nivel_tejido
0,p000,98.4,alto
1,p001,85.9,alto
2,p002,73.4,alto
3,p003,62.5,alto
4,p004,50.0,medio
5,p005,37.5,medio
6,p006,25.0,bajo
7,p007,12.5,bajo
8,p008,98.4,alto
9,p009,87.5,alto


In [44]:
# map NO puede: solo ve pct_tejido, no media_R
# apply SÍ: recibe la fila completa (axis=1 = "por filas")
df["revisar"] = df.apply(
    lambda fila: "sí" if (fila["pct_tejido"] > 50 and fila["media_R"] > 160) else "no",
    axis=1
)

In [45]:
df.sort_values("pct_tejido", ascending=False)

,slide_id,patch_id,pct_tejido,media_R,media_G,media_B,conservar,conservar_25,nivel_tejido,revisar
0,TCGA-00,p000,98.4,140.4,139.7,118.7,True,True,alto,no
8,TCGA-02,p008,98.4,130.5,135.2,134.0,True,True,alto,no
9,TCGA-02,p009,87.5,147.4,140.5,142.3,True,True,alto,no
1,TCGA-00,p001,85.9,132.0,146.3,125.3,True,True,alto,no
10,TCGA-02,p010,75.0,155.9,157.0,161.5,True,True,alto,no
2,TCGA-00,p002,73.4,149.7,159.0,143.5,True,True,alto,no
3,TCGA-00,p003,62.5,163.7,176.0,172.9,True,True,alto,sí
11,TCGA-02,p011,62.5,168.0,158.0,175.4,True,True,alto,sí
4,TCGA-01,p004,50.0,180.9,184.1,186.2,True,True,medio,no
5,TCGA-01,p005,37.5,199.5,203.7,202.2,False,True,medio,no


In [46]:
df.query("pct_tejido > 25 and media_R < 150") #'query' gana en legibilidad cuando la condición es larga o la compartes con alguien (se lee casi como español)

,slide_id,patch_id,pct_tejido,media_R,media_G,media_B,conservar,conservar_25,nivel_tejido,revisar
0,TCGA-00,p000,98.4,140.4,139.7,118.7,True,True,alto,no
1,TCGA-00,p001,85.9,132.0,146.3,125.3,True,True,alto,no
2,TCGA-00,p002,73.4,149.7,159.0,143.5,True,True,alto,no
8,TCGA-02,p008,98.4,130.5,135.2,134.0,True,True,alto,no
9,TCGA-02,p009,87.5,147.4,140.5,142.3,True,True,alto,no


In [47]:
# ~ = NOT: invierte la máscara → los parches que NO se conservan (descartados por el criterio oficial pct>=50)
df[~df["conservar"]][["patch_id", "pct_tejido", "conservar"]]   # pct < 50 → p005 (37.5), p006 (25.0), p007 (12.5)

,patch_id,pct_tejido,conservar
5,p005,37.5,False
6,p006,25.0,False
7,p007,12.5,False


In [48]:
def explorar_manifest(df):
    """Imprime una radiografía EDA del manifest: forma, tipos, resumen y % conservables."""
    print("Forma (filas, columnas):", df.shape)
    print("\nTipos por columna:")
    print(df.dtypes)
    print("\nResumen numérico:")
    print(df.describe())
    print("\n% de parches conservables:", round(df["conservar"].mean() * 100, 1))

explorar_manifest(df)

Forma (filas, columnas): (12, 10)

Tipos por columna:
slide_id         object
patch_id         object
pct_tejido      float64
media_R         float64
media_G         float64
media_B         float64
conservar          bool
conservar_25       bool
nivel_tejido     object
revisar          object
dtype: object

Resumen numérico:
       pct_tejido     media_R     media_G     media_B
count   12.000000   12.000000   12.000000   12.000000
mean    64.050000  168.275000  170.208333  167.208333
std     28.097088   33.335979   31.760280   36.068709
min     12.500000  130.500000  135.200000  118.700000
25%     46.875000  145.650000  144.850000  140.225000
50%     67.950000  159.800000  158.500000  167.200000
75%     86.300000  185.550000  189.000000  190.200000
max     98.400000  229.600000  236.100000  230.600000

% de parches conservables: 75.0


In [49]:
df.groupby("slide_id")["pct_tejido"].mean()

,pct_tejido
slide_id,
TCGA-00,80.05
TCGA-01,31.25
TCGA-02,80.85


In [50]:
df.groupby("slide_id")["pct_tejido"].agg(["mean", "count", "std", "min", "max"])

,mean,count,std,min,max
slide_id,,,,,
TCGA-00,80.05,4,15.526000,62.5,98.4
TCGA-01,31.25,4,16.137431,12.5,50.0
TCGA-02,80.85,4,15.526000,62.5,98.4


In [51]:
df.loc[3, "media_R"] = np.nan     # simulamos un hueco: al parche p003 le "falta" media_R

In [52]:
df.isnull()              # DataFrame de True/False: True donde hay NaN
df["media_R"].isnull()   # por columna: ¿qué filas tienen NaN en media_R?

,media_R
0,False
1,False
2,False
3,True
4,False
5,False
6,False
7,False
8,False
9,False


In [53]:
df.isnull().sum()        # cuántos NaN por columna

,0
slide_id,0
patch_id,0
pct_tejido,0
media_R,1
media_G,0
media_B,0
conservar,0
conservar_25,0
nivel_tejido,0
revisar,0


In [54]:
# El NaN ya está puesto arriba; aquí solo lo observamos
print(df.isnull().sum())                                  # media_R: 1
print("media global media_R:", df["media_R"].mean())      # promedia 11 valores (NaN omitido)
print(df.groupby("slide_id")["media_R"].agg(["mean", "count"]))   # TCGA-00 count=3, resto 4

slide_id        0
patch_id        0
pct_tejido      0
media_R         1
media_G         0
media_B         0
conservar       0
conservar_25    0
nivel_tejido    0
revisar         0
dtype: int64
media global media_R: 168.69090909090912
             mean  count
slide_id                
TCGA-00   140.700      3
TCGA-01   207.925      4
TCGA-02   150.450      4


In [55]:
df.dropna()                          # ELIMINAR las filas con algún NaN
df["media_R"].fillna(df["media_R"].mean())   # RELLENAR el hueco (aquí, con la media)

,media_R
0,140.400000
1,132.000000
2,149.700000
3,168.690909
4,180.900000
5,199.500000
6,221.700000
7,229.600000
8,130.500000
9,147.400000


In [56]:
# imputar media_R con la MEDIANA DE SU PROPIA LAMINILLA (lo más fino para tu estructura)
df["media_R"] = df["media_R"].fillna(
    df.groupby("slide_id")["media_R"].transform("median")
)
print(df.isnull().sum())    # media_R: 0 → hueco cerrado con criterio de grupo

slide_id        0
patch_id        0
pct_tejido      0
media_R         0
media_G         0
media_B         0
conservar       0
conservar_25    0
nivel_tejido    0
revisar         0
dtype: int64


In [57]:
df = df.rename(columns={"media_R": "media_r", "media_G": "media_g", "media_B": "media_b"})

In [58]:
df["conservar"] = df["conservar"].astype(bool)   # asegurar que es bool, no object
df["pct_tejido"] = df["pct_tejido"].astype(float) # asegurar float

In [59]:
def resumir_por_slide(df):
    """Agrega el manifest de parche-level a slide-level: una fila por laminilla.
    Devuelve un DataFrame con nº de parches, % tejido medio y firma de color media."""
    resumen = df.groupby("slide_id").agg(
        n_parches   = ("patch_id",   "count"),
        pct_tejido_medio = ("pct_tejido", "mean"),
        pct_tejido_std   = ("pct_tejido", "std"),
        media_r     = ("media_r",    "mean"),
        media_g     = ("media_g",    "mean"),
        media_b     = ("media_b",    "mean"),
    ).round(1)
    return resumen

manifest_slide = resumir_por_slide(df)
manifest_slide.to_csv("manifest_slide.csv")     # ← ojo: aquí SÍ queremos el índice
print(manifest_slide)

          n_parches  pct_tejido_medio  pct_tejido_std  media_r  media_g  \
slide_id                                                                  
TCGA-00           4              80.1            15.5    140.6    155.2   
TCGA-01           4              31.2            16.1    207.9    207.7   
TCGA-02           4              80.8            15.5    150.4    147.7   

          media_b  
slide_id           
TCGA-00     140.1  
TCGA-01     208.2  
TCGA-02     153.3  


In [60]:
manifest_slide = resumir_por_slide(df)
print(manifest_slide.shape)          # (3, 6) → índice slide_id aparte
manifest_slide.to_csv("manifest_slide.csv")
print(pd.read_csv("manifest_slide.csv").shape)   # (3, 7) → aquí slide_id ya es columna

(3, 6)
(3, 7)


## EDA de instrumento — Titanic
Practicar el gesto de EDA (faltantes, agregación por grupo) sobre un dataset tabular limpio,
con puente explícito a su equivalente en el manifest H&E.

In [61]:
import seaborn as sns
titanic = sns.load_dataset("titanic")   # columnas en minúscula: survived, pclass, sex, age, fare...
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [62]:
titanic.info()            # tipos + cuántos NO-nulos por columna
titanic.describe()        # resumen numérico
titanic.isnull().sum()    # inventario de faltantes por columna

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


### Manejo de faltantes
Decisión por columna, según *cuánto* falta y *por qué*:
- **`deck`** — 77 % faltante (MNAR: concentrado en 3ª clase) → **eliminar la columna** (demasiado hueca para imputar sin inventar).
- **`age`** — 20 % faltante → **imputar** con la mediana **por grupo** (`sex` × `pclass`), más un colchón global por si algún grupo quedara vacío.
- **`embarked`** — 2 faltantes → **eliminar esas 2 filas** (impacto despreciable).

In [63]:
# deck: 77% faltante (MNAR) → eliminar la columna
titanic = titanic.drop(columns="deck")

# age: 20% faltante → imputar por grupo (sex × pclass) + colchón global
titanic["age"] = titanic["age"].fillna(
    titanic.groupby(["sex", "pclass"])["age"].transform("median")
)
titanic["age"] = titanic["age"].fillna(titanic["age"].median())   # red de seguridad

# embarked: 2 faltantes → eliminar esas filas
titanic = titanic.dropna(subset=["embarked"])

print(titanic.isnull().sum())    # 0 en las columnas tratadas
print("shape:", titanic.shape)   # (889, 14): 891 − 2 filas, sin la columna deck

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64
shape: (889, 14)


### Agregación por grupo — tasa de supervivencia

In [64]:
# El mismo gesto que una "tasa de MSI-H por subgrupo": media de una variable 0/1 por grupo
print("Supervivencia por sexo:")
print(titanic.groupby("sex")["survived"].mean().round(3))
print("\nSupervivencia por clase:")
print(titanic.groupby("pclass")["survived"].mean().round(3))

Supervivencia por sexo:
sex
female    0.740
male      0.189
Name: survived, dtype: float64

Supervivencia por clase:
pclass
1    0.626
2    0.473
3    0.242
Name: survived, dtype: float64


### Puente clínico
El mismo gesto metodológico, sobre la cohorte H&E:
- Faltantes de `age`/`deck` ↔ faltantes de `MPP` / `tumor_pct` — y su **patrón** importa: un faltante MNAR (correlacionado con otra variable) sesga si se ignora.
- `groupby(...).survived.mean()` ↔ **tasa de MSI-H por subgrupo** y agregación **parche → laminilla → paciente**.
- Desbalance de clases (`survived`) ↔ **MSI-H ≈ 15 % del CRC**.

Titanic es el *maniquí*: se practica el gesto en limpio antes de aplicarlo al `manifest_slide.csv` real y a los *splits por paciente* del MIL.